# 🚀 Klint-32M: Full Scale Training on Google Colab

> **Klint** is an open research foundation architecture for generative financial time-series modeling.
> Author: Akhilesh Varma (`akhverm@gmail.com`)  
> Target Scale: **Klint-32M** (~30.6M parameters: 10 layers, 480 hidden dimension, 10 heads).

This notebook trains the **entire 32-Million parameter model** end-to-end on the complete dataset of **1,591,983 1-minute bars** (`SOL.npy`). All checkpoints are saved directly inside the Colab environment (`./checkpoints/`) with **zero Google Drive dependency**.

## 1. Hardware Detection & Accelerator Auto-Tuning
Detects available GPU (T4, V100, A100, or L4) and configures optimal precision (`bfloat16` or `float16`).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"GPU Detected: {gpu_name} ({total_vram:.2f} GB VRAM)")
    print(f"bfloat16 Acceleration Supported: {bf16_ok}")
else:
    print("WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> Select GPU (T4 or A100).")

## 2. Environment Setup & Architectural Verification
Ensures Python paths are configured, installs the package, and runs the test suite to verify 100% of invariant guarantees ($H \ge \max(O, C)$, $L \le \min(O, C)$, RoPE causal masking) before training.

In [ ]:
import sys, os
sys.path.insert(0, '/content/Klint-32M/src')
sys.path.insert(0, os.path.abspath('src'))

# If running in Colab from cloned repo:
# !git clone https://github.com/ak495867/Klint-32M.git
# %cd /content/Klint-32M

!pip install -e ".[dev]"
!pytest tests/ -v
!mkdir -p checkpoints data

## 3. Stage 1: Pre-training the Factor Tokenizer (RVQ Codebooks)
Train the multi-stream Residual Vector Quantizer across the 1.59M market bars to learn the discrete vocabulary:
* **512 Price-Path codes**
* **256 Range-Shape codes**
* **256 Activity codes**

Checkpoints are saved locally to `./checkpoints/tokenizer_best.pt`.

In [ ]:
!python scripts/train_tokenizer.py \
    --data_path data/SOL.npy \
    --epochs 15 \
    --batch_size 128 \
    --learning_rate 0.001 \
    --save_dir checkpoints

## 4. Stage 2: Discrete Token Caching (50x Training Acceleration)
Run the trained tokenizer over all 1.59M bars to encode them into discrete token matrices. Pre-caching tokens saves massive compute during Transformer training by eliminating redundant continuous feature extraction.

In [ ]:
!python scripts/cache_tokens.py \
    --data_path data/SOL.npy \
    --tokenizer_path checkpoints/tokenizer_best.pt \
    --output_path data/sol_tokens.pt \
    --batch_size 16384

## 5. Stage 3: Full 32-Million Parameter Foundation Model Training
Train the full **Klint-32M** causal autoregressive foundation model:
* **10 Transformer Layers, 480 Hidden Dimension, 10 Heads, 1920 FFN ($4\times$)**
* **256 Bars Context ($768$ factor tokens = ~4.2 hours of financial dynamics)**
* **Micro-batch: 8, Gradient Accumulation: 4 $\to$ Effective Batch Size: 32**
* **16x faster attention execution on Tesla T4 GPUs (~1 hour training)**

Checkpoints are saved locally to `./checkpoints/klint_32m_best.pt`.

In [ ]:
# Configure CUDA allocator to avoid fragmentation
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# High-speed configuration (Recommended for T4 GPU: ~1 hour full training)
# 3,000 steps at batch size 32 = ~15 full epochs over all 1.59M bars
!python scripts/train_klint32m.py \
    --tokens_path data/sol_tokens.pt \
    --save_dir checkpoints \
    --max_steps 3000 \
    --batch_size 8 \
    --grad_accum_steps 4 \
    --learning_rate 0.0003 \
    --warmup_steps 200 \
    --eval_interval 200 \
    --save_interval 500 \
    --context_bars 256 \
    --no_gradient_checkpointing

## 6. Stage 4: Package Full Model Bundle for Easy Download
Bundle the trained Tokenizer weights, Klint-32M foundation weights, and architecture configuration into a single standalone file: `./checkpoints/klint_32m_release.pt`.

In [ ]:
import os
import sys
sys.path.insert(0, '/content/Klint-32M/src')
sys.path.insert(0, os.path.abspath('src'))

import torch
from klint.models.klint_32m import Klint32M, KlintConfig
from klint.tokenizer.factor_tokenizer import FactorTokenizer

bundle_path = "checkpoints/klint_32m_release.pt"
print("Packaging unified release bundle...")

tok_ckpt = torch.load("checkpoints/tokenizer_best.pt", map_location="cpu")
model_ckpt = torch.load("checkpoints/klint_32m_best.pt", map_location="cpu")

torch.save({
    "architecture": "Klint-32M",
    "author": "Akhilesh Varma",
    "config": model_ckpt["config"],
    "tokenizer_state_dict": tok_ckpt,
    "model_state_dict": model_ckpt["model_state_dict"],
    "val_loss": model_ckpt.get("val_loss", None),
}, bundle_path)

bundle_size_mb = os.path.getsize(bundle_path) / (1024 * 1024)
print(f"Release bundle created: {bundle_path} ({bundle_size_mb:.2f} MB)")

# Optional: Trigger browser download straight to your local PC
# from google.colab import files
# files.download(bundle_path)

## 7. Stage 5: Generative Trajectory Sampling & Candlestick Visualization
Condition on real market bars, autoregressively sample 100 future financial bars using Klint-32M, decode them with the structural geometric decoder, and plot the candlestick trajectory.

In [ ]:
import os
import sys
sys.path.insert(0, '/content/Klint-32M/src')
sys.path.insert(0, os.path.abspath('src'))

import numpy as np
import matplotlib.pyplot as plt
import torch

from klint.tokenizer.factor_tokenizer import FactorTokenizer
from klint.tokenizer.geometric_decoder import GeometricDecoder
from klint.models.klint_32m import Klint32M, KlintConfig
from klint.data.validator import validate_ohlcv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load trained bundle
bundle = torch.load("checkpoints/klint_32m_release.pt", map_location=device)
tokenizer = FactorTokenizer().to(device)
tokenizer.load_state_dict(bundle["tokenizer_state_dict"])
tokenizer.eval()

config = bundle["config"]
model = Klint32M(config).to(device)
model.load_state_dict(bundle["model_state_dict"])
model.eval()
decoder = GeometricDecoder()

# 2. Load prompt tokens from real dataset
tokens_data = torch.load("data/sol_tokens.pt", map_location="cpu")
token_matrix = tokens_data["token_matrix"]

# Use last 50 bars (150 tokens) as prompt
prompt_tokens = token_matrix[-50:].view(1, -1).to(device)
num_future_bars = 100

print(f"Generating {num_future_bars} synthetic bars in causal sequence (Price -> Range -> Activity)...")
gen_tokens = model.generate_tokens(prompt_tokens, num_bars=num_future_bars, temperature=0.8, top_k=40)
new_tokens = gen_tokens[:, 150:]  # Exclude prompt

# Decode tokens to factors
p_tok, r_tok, a_tok = tokenizer.deinterleave(new_tokens)
rec_p, rec_r, rec_a = tokenizer.decode_tokens(p_tok, r_tok, a_tok)

# Geometric decoding with invariant guarantees
anchor_p = float(tokens_data.get("anchor_price", 100.0))
synth_ohlcv = decoder(rec_p, rec_r, rec_a, anchor_price=anchor_p)
synth_bars = synth_ohlcv.squeeze(0).detach().cpu().numpy()

is_valid, report = validate_ohlcv(synth_bars)
print(f"Generated Candles Validity: {is_valid} | Violations: {report['total_violations']}")
assert is_valid, "Generated candles violated geometry!"

# 3. Plot Candlesticks
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

for i in range(len(synth_bars)):
    o, h, l, c, v = synth_bars[i]
    color = '#26a69a' if c >= o else '#ef5350'
    # Draw High-Low wick
    ax1.plot([i, i], [l, h], color=color, linewidth=1.2)
    # Draw Open-Close body
    body_bottom = min(o, c)
    body_height = max(abs(c - o), 0.001)
    ax1.add_patch(plt.Rectangle((i - 0.35, body_bottom), 0.7, body_height, color=color, alpha=0.9))
    # Volume bar
    ax2.bar(i, v, color=color, width=0.7, alpha=0.8)

ax1.set_title("Klint-32M: Synthetic Generated Market Trajectory (100 Bars)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Price ($)", fontsize=12)
ax1.grid(True, alpha=0.2)
ax2.set_ylabel("Volume", fontsize=12)
ax2.set_xlabel("Bar Step (t)", fontsize=12)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("checkpoints/synthetic_market_trajectory.png", dpi=150)
plt.show()
print("Plot saved to checkpoints/synthetic_market_trajectory.png")